# CLINC150 Single-Domain OOS Download

This notebook is intended for VS Code with a Colab runtime. It mounts Google Drive, downloads CLINC single-domain OOS banking and credit_cards subsets, and exports them to Drive.

Primary source: `Salesforce/dialogstudio` configs. This dataset is gated on Hugging Face, so you must accept the dataset conditions and authenticate. If that source is unavailable, the notebook can fall back to the public `clinc/clinc_oos` dataset and reconstruct domain subsets from the official CLINC domain intent list.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip -q install -U datasets huggingface_hub pandas pyarrow matplotlib

In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from datasets import DatasetDict, load_dataset
from huggingface_hub import login, notebook_login

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "FinDisputeEval"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUTPUT_DIR = PROJECT_DIR / "dataset" / "interim" / "clinc150_single_domain_oos"
CACHE_DIR = PROJECT_DIR / "dataset" / "cache" / "huggingface"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")

# SOURCE_MODE options:
# - "auto": try gated DialogStudio first, then fall back to public clinc/clinc_oos
# - "dialogstudio": require Salesforce/dialogstudio and fail if gated access is unavailable
# - "clinc_oos_fallback": skip DialogStudio and reconstruct from public clinc/clinc_oos
SOURCE_MODE = "auto"
AUTHENTICATE_HF = True

SUBSETS = {
    "banking": "CLINC-Single-Domain-OOS-banking",
    "credit_cards": "CLINC-Single-Domain-OOS-credit_cards",
}

DOWNLOAD_FULL_CLINC_PLUS = False

hf_token = None
if AUTHENTICATE_HF:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = os.environ.get("HF_TOKEN")

    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        HF_AUTH_KWARGS = {"token": True}
        print("Hugging Face login complete using HF_TOKEN.")
    else:
        print("No Colab secret/environment variable named HF_TOKEN was found.")
        print("A login prompt will open. Use a Hugging Face read token after accepting Salesforce/dialogstudio access terms.")
        try:
            notebook_login()
            HF_AUTH_KWARGS = {"token": True}
        except Exception as exc:
            print(f"HF interactive login was skipped or failed: {exc}")
            HF_AUTH_KWARGS = {}
else:
    HF_AUTH_KWARGS = {}

print(f"Output directory: {OUTPUT_DIR}")
print(f"HF cache directory: {CACHE_DIR}")

In [ ]:
CLINC_DOMAIN_INTENTS = {
    "banking": [
        "freeze_account", "routing", "pin_change", "bill_due", "pay_bill",
        "account_blocked", "interest_rate", "min_payment", "bill_balance",
        "transfer", "order_checks", "balance", "spending_history",
        "transactions", "report_fraud",
    ],
    "credit_cards": [
        "replacement_card_duration", "expiration_date", "damaged_card",
        "improve_credit_score", "report_lost_card", "card_declined",
        "credit_limit_change", "apr", "redeem_rewards", "credit_limit",
        "rewards_balance", "application_status", "credit_score", "new_card",
        "international_fees",
    ],
}


def load_dialogstudio_config(config_name: str) -> DatasetDict:
    """Load a DialogStudio config, retrying with trust_remote_code for older dataset scripts."""
    try:
        return load_dataset("Salesforce/dialogstudio", config_name, **HF_AUTH_KWARGS)
    except Exception as exc:
        print(f"Initial load failed for {config_name}: {exc}")
        print("Retrying with trust_remote_code=True ...")
        return load_dataset("Salesforce/dialogstudio", config_name, trust_remote_code=True, **HF_AUTH_KWARGS)


def load_clinc_oos_domain_subset(domain_name: str) -> DatasetDict:
    """Reconstruct a domain subset from public clinc/clinc_oos plus split.

    This is not the DialogStudio converted schema. It preserves CLINC text/intent data,
    keeps the selected domain intents, and includes the official OOS label.
    """
    if domain_name not in CLINC_DOMAIN_INTENTS:
        raise ValueError(f"Unsupported CLINC domain: {domain_name}")

    full = load_dataset("clinc/clinc_oos", "plus")
    domain_intents = set(CLINC_DOMAIN_INTENTS[domain_name])
    keep_intents = domain_intents | {"oos"}
    subset_splits = {}

    for split_name, split_ds in full.items():
        label_col = "intent" if "intent" in split_ds.column_names else "label"
        label_feature = split_ds.features[label_col]
        label_names = label_feature.names

        def add_domain_fields(batch):
            ids = [int(value) for value in batch[label_col]]
            names = [label_names[value] for value in ids]
            return {
                "intent_id": ids,
                "intent_name": names,
                "clinc_domain": [domain_name if name in domain_intents else "oos" for name in names],
                "is_oos": [name == "oos" for name in names],
            }

        enriched = split_ds.map(add_domain_fields, batched=True)
        subset_splits[split_name] = enriched.filter(lambda row: row["intent_name"] in keep_intents)

    return DatasetDict(subset_splits)


def load_single_domain_subset(subset_key: str, config_name: str):
    if SOURCE_MODE in {"auto", "dialogstudio"}:
        try:
            ds = load_dialogstudio_config(config_name)
            return ds, f"Salesforce/dialogstudio/{config_name}"
        except Exception as exc:
            if SOURCE_MODE == "dialogstudio":
                raise
            print(f"DialogStudio load failed for {config_name}: {exc}")
            print("Falling back to public clinc/clinc_oos plus domain reconstruction.")

    ds = load_clinc_oos_domain_subset(subset_key)
    return ds, f"clinc/clinc_oos/plus::{subset_key}+oos"


def export_dataset_dict(ds: DatasetDict, subset_key: str, source_id: str) -> dict:
    subset_dir = OUTPUT_DIR / subset_key
    subset_dir.mkdir(parents=True, exist_ok=True)

    ds.save_to_disk(str(subset_dir / "hf_dataset"))

    manifest = {
        "subset_key": subset_key,
        "source_id": source_id,
        "output_dir": str(subset_dir),
        "splits": {},
    }

    for split_name, split_ds in ds.items():
        split_stem = subset_dir / split_name
        manifest["splits"][split_name] = {
            "rows": len(split_ds),
            "features": list(split_ds.features.keys()),
            "jsonl": str(split_stem.with_suffix(".jsonl")),
            "csv": str(split_stem.with_suffix(".csv")),
            "parquet": str(split_stem.with_suffix(".parquet")),
        }

        split_ds.to_json(str(split_stem.with_suffix(".jsonl")), orient="records", lines=True, force_ascii=False)
        split_ds.to_csv(str(split_stem.with_suffix(".csv")))
        split_ds.to_parquet(str(split_stem.with_suffix(".parquet")))

    (subset_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest


def preview_dataset_dict(ds: DatasetDict, subset_key: str, n: int = 3) -> None:
    print(f"\n{subset_key}")
    for split_name, split_ds in ds.items():
        print(f"  {split_name}: {len(split_ds):,} rows | columns: {list(split_ds.features.keys())}")
        if len(split_ds) > 0:
            display(split_ds.select(range(min(n, len(split_ds)))).to_pandas())

In [ ]:
manifests = {}
downloaded = {}

for subset_key, config_name in SUBSETS.items():
    print(f"\nLoading {subset_key} ({config_name})")
    ds, source_id = load_single_domain_subset(subset_key, config_name)
    downloaded[subset_key] = ds
    preview_dataset_dict(ds, subset_key)
    manifests[subset_key] = export_dataset_dict(ds, subset_key, source_id)

print("\nDone. Exported subsets:")
for subset_key, manifest in manifests.items():
    print(f"- {subset_key}: {manifest['output_dir']}")

In [ ]:
if DOWNLOAD_FULL_CLINC_PLUS:
    print("Loading full clinc_oos plus config ...")
    full_clinc = load_dataset("clinc/clinc_oos", "plus")
    preview_dataset_dict(full_clinc, "clinc_oos_plus")
    manifests["clinc_oos_plus"] = export_dataset_dict(full_clinc, "clinc_oos_plus", "clinc/clinc_oos/plus")

In [ ]:
summary_rows = []

for subset_key, manifest in manifests.items():
    for split_name, split_info in manifest["splits"].items():
        summary_rows.append({
            "subset": subset_key,
            "split": split_name,
            "rows": split_info["rows"],
            "features": ", ".join(split_info["features"]),
            "jsonl": split_info["jsonl"],
            "csv": split_info["csv"],
            "parquet": split_info["parquet"],
        })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "download_summary.csv"
summary_df.to_csv(summary_path, index=False)

display(summary_df)
print(f"Summary saved to: {summary_path}")

## FinDisputeEval EDA

Use this section after the download/export cells. For the FinDispute project, CLINC150 is used as a normal banking / out-of-scope boundary dataset, not as a regulatory or complaint corpus.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import DatasetDict, load_from_disk

EDA_DIR = PROJECT_DIR / "outputs" / "data_pipeline" / "clinc150_oos_eda" / "eda_v01" / f"run_{RUN_ID}_colab"
EDA_DIR.mkdir(parents=True, exist_ok=True)

TEXT_CANDIDATES = ["text", "utterance", "query", "sentence", "prompt", "input"]
INTENT_CANDIDATES = ["intent_name", "intent", "label", "category", "dialog_act"]

KEYWORD_GROUPS = {
    "dispute_core": [r"\bdisput(e|ed|ing)?\b", r"\bchargeback\b", r"billing error", r"wrong charge"],
    "unauthorized_fraud": [r"unauthori[sz]ed", r"\bfraud\b", r"stolen", r"lost card", r"didn'?t authorize", r"do not recognize", r"don'?t recognize"],
    "credit_card_boundary": [r"credit card", r"\bapr\b", r"credit limit", r"rewards", r"card declined", r"new card"],
    "debit_eft_p2p_boundary": [r"debit card", r"\bach\b", r"\beft\b", r"zelle", r"wire", r"transfer", r"routing"],
    "money_movement": [r"payment", r"refund", r"merchant", r"transaction", r"balance", r"deposit", r"withdraw"],
    "escalation_stress": [r"complain", r"supervisor", r"manager", r"lawsuit", r"regulator", r"cfpb", r"angry", r"upset"],
}


def safe_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    if isinstance(value, dict):
        preferred_keys = ["text", "utterance", "content", "user", "system", "speaker", "original user side information"]
        pieces = [safe_text(value.get(key)) for key in preferred_keys if key in value]
        if not pieces:
            pieces = [safe_text(v) for v in value.values()]
        return " ".join(piece for piece in pieces if piece).strip()
    if isinstance(value, (list, tuple, np.ndarray)):
        return " ".join(safe_text(item) for item in value if safe_text(item)).strip()
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for column in candidates:
        if column in df.columns:
            return column
    return None


def datasetdict_to_findispute_frame(ds: DatasetDict, subset_key: str) -> pd.DataFrame:
    frames = []
    for split_name, split_ds in ds.items():
        df = split_ds.to_pandas()
        text_col = first_existing_column(df, TEXT_CANDIDATES)
        intent_col = first_existing_column(df, INTENT_CANDIDATES)

        if text_col:
            df["utterance"] = df[text_col].map(safe_text)
        else:
            fallback_text_cols = [col for col in ["log", "turns", "dialog", "original dialog info"] if col in df.columns]
            df["utterance"] = df[fallback_text_cols].apply(lambda row: " ".join(safe_text(v) for v in row), axis=1) if fallback_text_cols else ""

        df["intent_label"] = df[intent_col].map(safe_text) if intent_col else "unknown"
        if "intent_name" in df.columns:
            df["intent_label"] = df["intent_name"].map(safe_text)

        if "is_oos" in df.columns:
            df["is_oos_norm"] = df["is_oos"].astype(str).str.lower().isin(["true", "1", "yes"])
        else:
            df["is_oos_norm"] = df["intent_label"].str.lower().str.contains(r"(^|[_ -])oos($|[_ -])|out.?of.?scope", regex=True, na=False)

        df["source_subset"] = subset_key
        df["split"] = split_name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


if "downloaded" not in globals() or not downloaded:
    downloaded = {}
    for subset_key in SUBSETS:
        hf_dataset_path = OUTPUT_DIR / subset_key / "hf_dataset"
        if hf_dataset_path.exists():
            downloaded[subset_key] = load_from_disk(str(hf_dataset_path))

if not downloaded:
    raise RuntimeError("No downloaded datasets found. Run the download/export cells first.")

clinc_eda_df = pd.concat(
    [datasetdict_to_findispute_frame(ds, subset_key) for subset_key, ds in downloaded.items() if subset_key in SUBSETS],
    ignore_index=True,
)

clinc_eda_df["utterance_norm"] = clinc_eda_df["utterance"].fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
clinc_eda_df["char_count"] = clinc_eda_df["utterance_norm"].str.len()
clinc_eda_df["word_count"] = clinc_eda_df["utterance_norm"].str.split().map(len)

for group_name, patterns in KEYWORD_GROUPS.items():
    regex = "|".join(patterns)
    clinc_eda_df[f"kw_{group_name}"] = clinc_eda_df["utterance_norm"].str.contains(regex, regex=True, na=False)

keyword_cols = [f"kw_{group_name}" for group_name in KEYWORD_GROUPS]
clinc_eda_df["keyword_hit_count"] = clinc_eda_df[keyword_cols].sum(axis=1)
clinc_eda_df["any_findispute_keyword"] = clinc_eda_df["keyword_hit_count"] > 0


def assign_findispute_role(row) -> str:
    if row["is_oos_norm"]:
        return "oos_negative_boundary"
    if row["kw_dispute_core"] or row["kw_unauthorized_fraud"]:
        return "near_scope_dispute_boundary_probe"
    if row["source_subset"] == "credit_cards":
        return "credit_card_servicing_non_dispute"
    if row["source_subset"] == "banking":
        return "banking_servicing_non_dispute"
    return "other"


clinc_eda_df["findispute_role"] = clinc_eda_df.apply(assign_findispute_role, axis=1)

normalized_path = EDA_DIR / "findispute_clinc_normalized_rows.parquet"
clinc_eda_df.to_parquet(normalized_path, index=False)
clinc_eda_df.to_csv(EDA_DIR / "findispute_clinc_normalized_rows.csv", index=False)

print(f"Rows prepared for FinDispute EDA: {len(clinc_eda_df):,}")
print(f"Saved normalized rows to: {normalized_path}")
display(clinc_eda_df[["source_subset", "split", "intent_label", "is_oos_norm", "findispute_role", "utterance"]].head(10))

In [ ]:
split_summary = (
    clinc_eda_df.groupby(["source_subset", "split"], dropna=False)
    .agg(
        rows=("utterance", "size"),
        oos_rows=("is_oos_norm", "sum"),
        avg_words=("word_count", "mean"),
        p95_words=("word_count", lambda s: s.quantile(0.95)),
        keyword_rows=("any_findispute_keyword", "sum"),
    )
    .reset_index()
)
split_summary["oos_rate"] = split_summary["oos_rows"] / split_summary["rows"]
split_summary["keyword_rate"] = split_summary["keyword_rows"] / split_summary["rows"]

intent_distribution = (
    clinc_eda_df.groupby(["source_subset", "split", "intent_label", "findispute_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source_subset", "split", "rows"], ascending=[True, True, False])
)

keyword_summary = (
    clinc_eda_df.melt(
        id_vars=["source_subset", "split", "intent_label"],
        value_vars=keyword_cols,
        var_name="keyword_group",
        value_name="hit",
    )
    .query("hit")
    .groupby(["source_subset", "split", "keyword_group"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source_subset", "split", "rows"], ascending=[True, True, False])
)

role_summary = (
    clinc_eda_df.groupby(["source_subset", "findispute_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["source_subset", "rows"], ascending=[True, False])
)

split_summary.to_csv(EDA_DIR / "findispute_clinc_split_summary.csv", index=False)
intent_distribution.to_csv(EDA_DIR / "findispute_clinc_intent_distribution.csv", index=False)
keyword_summary.to_csv(EDA_DIR / "findispute_clinc_keyword_summary.csv", index=False)
role_summary.to_csv(EDA_DIR / "findispute_clinc_role_summary.csv", index=False)

display(split_summary)
display(role_summary)
display(intent_distribution.groupby(["source_subset", "intent_label"], as_index=False)["rows"].sum().sort_values("rows", ascending=False).head(30))
display(keyword_summary.head(30))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

split_pivot = split_summary.pivot(index="split", columns="source_subset", values="rows").fillna(0)
split_pivot.plot(kind="bar", ax=axes[0, 0], title="Rows by split and subset")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("rows")

oos_pivot = split_summary.pivot(index="split", columns="source_subset", values="oos_rate").fillna(0)
oos_pivot.plot(kind="bar", ax=axes[0, 1], title="OOS rate by split")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("OOS rate")

role_pivot = role_summary.pivot(index="findispute_role", columns="source_subset", values="rows").fillna(0)
role_pivot.plot(kind="barh", ax=axes[1, 0], title="FinDispute role coverage")
axes[1, 0].set_xlabel("rows")
axes[1, 0].set_ylabel("")

if len(keyword_summary) > 0:
    keyword_plot = keyword_summary.groupby("keyword_group", as_index=True)["rows"].sum().sort_values()
    keyword_plot.plot(kind="barh", ax=axes[1, 1], title="FinDispute keyword hits")
else:
    axes[1, 1].text(0.5, 0.5, "No keyword hits", ha="center", va="center")
    axes[1, 1].set_title("FinDispute keyword hits")
axes[1, 1].set_xlabel("rows")
axes[1, 1].set_ylabel("")

plt.tight_layout()
plot_path = EDA_DIR / "findispute_clinc_eda_overview.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
print(f"Saved plot to: {plot_path}")
plt.show()

In [ ]:
RANDOM_STATE = 42

sample_specs = [
    (
        "banking_normal_negative",
        (clinc_eda_df["source_subset"].eq("banking") & ~clinc_eda_df["is_oos_norm"] & ~clinc_eda_df["any_findispute_keyword"]),
        30,
    ),
    (
        "credit_card_servicing_boundary",
        (clinc_eda_df["source_subset"].eq("credit_cards") & ~clinc_eda_df["is_oos_norm"]),
        30,
    ),
    (
        "dispute_adjacent_probe",
        (~clinc_eda_df["is_oos_norm"] & clinc_eda_df["any_findispute_keyword"]),
        20,
    ),
    (
        "oos_negative_boundary",
        clinc_eda_df["is_oos_norm"],
        20,
    ),
]

sample_parts = []
used_indexes = set()

for bucket_name, mask, target_n in sample_specs:
    pool = clinc_eda_df.loc[mask & ~clinc_eda_df.index.isin(used_indexes)].copy()
    if pool.empty:
        print(f"No rows available for bucket: {bucket_name}")
        continue
    n = min(target_n, len(pool))
    sampled = pool.sample(n=n, random_state=RANDOM_STATE).copy()
    sampled["seed_bucket"] = bucket_name
    used_indexes.update(sampled.index.tolist())
    sample_parts.append(sampled)

if sample_parts:
    seed_sample = pd.concat(sample_parts).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    seed_sample = clinc_eda_df.sample(n=min(100, len(clinc_eda_df)), random_state=RANDOM_STATE).copy()
    seed_sample["seed_bucket"] = "fallback_random"

export_cols = [
    "seed_bucket", "source_subset", "split", "intent_label", "is_oos_norm",
    "findispute_role", "keyword_hit_count", "utterance",
]
seed_sample_path = EDA_DIR / "findispute_clinc_eval_seed_sample.csv"
seed_sample[export_cols].to_csv(seed_sample_path, index=False)

print(f"Saved FinDispute CLINC eval seed sample: {seed_sample_path}")
print(f"Sample rows: {len(seed_sample):,}")
display(seed_sample[export_cols].head(30))

## Expected Output Structure

```text
dataset/interim/clinc150_single_domain_oos/
  banking/
    hf_dataset/
    manifest.json
    train.jsonl / train.csv / train.parquet
    ...
  credit_cards/
    hf_dataset/
    manifest.json
    train.jsonl / train.csv / train.parquet
    ...
  download_summary.csv

outputs/data_pipeline/clinc150_oos_eda/eda_v01/<run_id>/
  findispute_clinc_normalized_rows.parquet
  findispute_clinc_eda_overview.png
  findispute_clinc_eval_seed_sample.csv
```
